In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import paths   # input/output locations: set SN2024VJM_DATA / SN2024VJM_EXTERNAL (see README)
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib widget

In [ ]:
t0 = 60572.15
def rise_power_law(t, a, t_exp, n):
    """
    Power law rise model: flux = a * (t - t_exp)^n for t > t_exp, 0 otherwise
    This ensures flux stays at zero before explosion and rises as a power law after
    
    Parameters:
    -----------
    t : array
        Time array (MJD)
    a : float
        Normalization constant (flux scale)
    t_exp : float
        Explosion time (MJD)
    n : float
        Power law index (should be positive for rising light curve)
    
    Returns:
    --------
    flux : array
        Flux values
    """
    return np.where(t - t_exp > 0, a * ((t - t_exp)**n), 0)

def estimate_explosion_time_from_q_band():
    """
    Estimate explosion time using power law fit to q band flux from both data sources
    """
    print("\n" + "="*60)
    print("EXPLOSION TIME ESTIMATION FROM Q BAND")
    print("="*60)
    
    # Initialize empty list to store all q-band data
    all_q_data = []
    
    # Load data from new_phot_bg.csv
    try:
        bg_new = pd.read_csv(paths.PHOTOMETRY / 'new_phot_bg.csv', sep=' ')
        print("Successfully loaded new_phot_bg.csv")
        
        # Filter for q band detections only (using cond2: FNU_ZOGY > 3*FNUERR_ZOGY)
        q_detections_bg = bg_new[
            (bg_new['FILTER'] == 'q') & 
            (bg_new['FNU_ZOGY'] > 3 * bg_new['FNUERR_ZOGY']) &
            (bg_new['MAG_ZOGY'] < 99)  # Valid magnitude
        ].copy()
        
        print(f"Found {len(q_detections_bg)} q-band detections in new_phot_bg.csv")
        
        # Add to combined data
        for _, row in q_detections_bg.iterrows():
            all_q_data.append({
                'mjd': row['MJD-OBS'],
                'flux': row['FNU_ZOGY'],
                'flux_err': row['FNUERR_ZOGY'],
                'filter': 'q',
                'source': 'new_phot_bg'
            })
            
    except Exception as e:
        print(f"Could not load new_phot_bg.csv: {e}")
    
    # Load data from ML.csv
    try:
        ml_data = pd.read_csv(paths.PHOTOMETRY / 'ML.csv')
        print("Successfully loaded ML.csv")
        
        # Filter for q band detections with good SNR
        q_detections_ml = ml_data[
            (ml_data['Filter'] == 'q') & 
            (ml_data['SNR_zogy'] > 3) &
            (ml_data['Mag_zogy'] < 99)  # Valid magnitude
        ].copy()
        
        print(f"Found {len(q_detections_ml)} q-band detections in ML.csv")
        
        # Add to combined data
        for _, row in q_detections_ml.iterrows():
            all_q_data.append({
                'mjd': row['MJD'],
                'flux': row['Flux_zogy'],
                'flux_err': row['Fluxerr_zogy'],
                'filter': 'q',
                'source': 'ML'
            })
            
    except Exception as e:
        print(f"Could not load ML.csv: {e}")
    
    if len(all_q_data) == 0:
        print("No q-band data found in either file")
        return None
    
    # Convert to DataFrame
    q_combined = pd.DataFrame(all_q_data)
    print(f"Total combined q-band detections: {len(q_combined)}")
    print(f"From new_phot_bg.csv: {len([d for d in all_q_data if d['source'] == 'new_phot_bg'])}")
    print(f"From ML.csv: {len([d for d in all_q_data if d['source'] == 'ML'])}")
    
    if len(q_combined) < 4:
        print("Not enough q-band detections for power law fit")
        return None
    
    # Simple binning function
    def bin_data_simple(df, bin_size=0.3):
        """Simple binning by time proximity"""
        binned_data = []
        df_sorted = df.sort_values('mjd')
        used_indices = set()
        
        for i, row in df_sorted.iterrows():
            if i in used_indices:
                continue
                
            # Find nearby points
            time_diff = np.abs(df_sorted['mjd'] - row['mjd'])
            nearby_mask = time_diff <= bin_size
            nearby_indices = df_sorted[nearby_mask].index.tolist()
            
            if any(idx in used_indices for idx in nearby_indices):
                continue
                
            nearby_data = df_sorted.loc[nearby_indices]
            
            if len(nearby_data) == 1:
                binned_row = row.copy()
            else:
                # Weighted average for flux
                weights = 1.0 / nearby_data['flux_err']**2
                weighted_flux = np.average(nearby_data['flux'], weights=weights)
                weighted_flux_err = 1.0 / np.sqrt(np.sum(weights))
                weighted_time = np.average(nearby_data['mjd'], weights=weights)
                
                binned_row = row.copy()
                binned_row['flux'] = weighted_flux
                binned_row['flux_err'] = weighted_flux_err
                binned_row['mjd'] = weighted_time
                binned_row['source'] = 'binned_' + '_'.join(nearby_data['source'].unique())
            
            binned_data.append(binned_row)
            used_indices.update(nearby_indices)
        
        return pd.DataFrame(binned_data)
    
    # Apply time cutoff first - only use data until MJD 60572.26
    time_cutoff = 60572.26
    q_filtered = q_combined[q_combined['mjd'] <= time_cutoff].copy()
    print(f"After time cutoff (MJD <= {time_cutoff}): {len(q_filtered)} q-band points")
    
    # Bin each data source separately (don't combine BG and ML)
    bg_data = q_filtered[q_filtered['source'] == 'new_phot_bg'].copy()
    ml_data = q_filtered[q_filtered['source'] == 'ML'].copy()
    
    binned_data = []
    
    # Bin BG data separately
    if len(bg_data) > 0:
        bg_binned = bin_data_simple(bg_data, bin_size=0.01)
        binned_data.append(bg_binned)
        print(f"BG data after binning: {len(bg_binned)} points")
    
    # Bin ML data separately  
    if len(ml_data) > 0:
        ml_binned = bin_data_simple(ml_data, bin_size=0.3)
        binned_data.append(ml_binned)
        print(f"ML data after binning: {len(ml_binned)} points")
    
    # Combine the separately binned data
    if binned_data:
        q_binned = pd.concat(binned_data, ignore_index=True)
        q_binned = q_binned.sort_values('mjd')
    else:
        print("No data available after filtering and binning")
        return None
    
    print(f"Total after separate binning: {len(q_binned)} q-band points")
    
    # Use flux directly - no conversion needed!
    flux = q_binned['flux'].values
    flux_err = q_binned['flux_err'].values
    times = q_binned['mjd'].values
    
    # Find peak for selecting early data (highest flux = lowest magnitude)
    peak_idx = q_binned['flux'].idxmax()
    peak_time = q_binned.loc[peak_idx, 'mjd']
    
    # Use only early data (before peak + 2 days) AND not more than 15 days before B-band maximum
    b_band_max = 60574.93  # B-band maximum time (t0)
    early_mask = (times <= peak_time + 2) & (times >= b_band_max - 15) & (times <= time_cutoff)
    times_early = times[early_mask]
    flux_early = flux[early_mask]
    flux_err_early = flux_err[early_mask]
    
    print(f"Using {len(times_early)} early points for fitting (before MJD {time_cutoff})")
    print(f"Time range: MJD {times_early.min():.1f} to {times_early.max():.1f}")
    print(f"Time range relative to B-band max: {times_early.min() - b_band_max:.1f} to {times_early.max() - b_band_max:.1f} days")
    
    # Power law fit
    print("\n--- Power Law Fit ---")
    try:
        # Initial guess: explosion ~5-10 days before first detection
        t_exp_guess = times_early.min() - 7
        a_guess = flux_early.max() / ((times_early.max() - t_exp_guess)**2)
        n_guess = 2.0
        
        # Bounds: a > 0, t_exp before first detection, 0 < n < 5
        bounds = ([1e-10, times_early.min() - 20, 0.1], 
                 [np.inf, times_early.min(), 5.0])
        
        popt_power, pcov_power = curve_fit(
            rise_power_law, times_early, flux_early, 
            sigma=flux_err_early, 
            p0=[a_guess, t_exp_guess, n_guess],
            bounds=bounds
        )
        
        a_fit, t_exp_power, n_fit = popt_power
        t_exp_err = np.sqrt(pcov_power[1, 1])
        n_fit_err = np.sqrt(pcov_power[2, 2])
        
        print(f"Power law fit: flux = {a_fit:.2e} * (t - {t_exp_power:.2f})^{n_fit:.2f}")
        print(f"Explosion time: MJD {t_exp_power:.2f} ± {t_exp_err:.2f}")
        print(f"Power law index: {n_fit:.2f} ± {n_fit_err:.2f}")
        
        # Calculate chi-squared
        model_flux = rise_power_law(times_early, *popt_power)
        chi2_power = np.sum((flux_early - model_flux)**2 / flux_err_early**2)
        dof_power = len(times_early) - 3
        chi2_red_power = chi2_power / dof_power
        print(f"Chi-squared: {chi2_power:.2f}")
        print(f"Degrees of freedom: {dof_power}")
        print(f"Reduced chi2: {chi2_red_power:.2f}")
        
        power_success = True
        
    except Exception as e:
        print(f"Power law fit failed: {e}")
        power_success = False
        t_exp_power = None
    
    # Final result
    if power_success:
        t_exp_best = t_exp_power
        method_used = "Power Law"
    else:
        print("Power law fit failed")
        return None
    
    print(f"\n--- Final Result ---")
    print(f"Best explosion time estimate: MJD {t_exp_best:.2f} ± {t_exp_err:.2f} ({method_used})")
    print(f"Days before first detection: {times_early.min() - t_exp_best:.1f}")
    print(f"Days before B-band maximum: {t0 - t_exp_best:.1f}")
    
    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Plot: Flux vs time with power law fit
    if power_success:
        # Color points by source - keep BG and ML separate
        sources = q_binned.loc[q_binned['mjd'].isin(times_early), 'source']
        colors = {'new_phot_bg': '#FFD700', 'ML': '#FF6B6B'}
        
        for source in sources.unique():
            mask = sources == source
            times_source = times_early[mask.values]
            flux_source = flux_early[mask.values]
            flux_err_source = flux_err_early[mask.values]
            
            if len(times_source) > 0:
                color = colors.get(source, '#808080')
                label = f'{source} (n={len(times_source)})'
                ax.errorbar(times_source, flux_source, yerr=flux_err_source, 
                           fmt='o', color=color, markersize=8, capsize=3, label=label)
        
        t_fine = np.linspace(t_exp_power - 2, times_early.max() + 2, 300)
        flux_model = rise_power_law(t_fine, *popt_power)
        
        ax.plot(t_fine, flux_model, 'r-', linewidth=2, 
                label=f'Power law: $a(t-{t_exp_power:.1f})^{{{n_fit:.2f}}}$')
        ax.axvline(t_exp_power, color='orange', linestyle='--', linewidth=2,
                   label=f'Explosion: MJD {t_exp_power:.1f}')
        
        ax.set_xlabel('MJD')
        ax.set_ylabel('Flux')
        ax.set_title(f'Q-band Power Law Explosion Time Fit (chi2/dof = {chi2_red_power:.2f})\nData until MJD {time_cutoff:.2f} - BG and ML binned separately')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_yscale('log')
    
    plt.tight_layout()
    plt.savefig(paths.OUTPUTS / 'explosion_time_q_band_powerlaw.png', 
                dpi=300, bbox_inches='tight')
    print(f"Plot saved as explosion_time_q_band_powerlaw.png")
    
    plt.show()
    
    return t_exp_best

# Calculate explosion time from q band
print("\n" + "="*80)
print("SEPARATE ANALYSIS: EXPLOSION TIME ESTIMATION")
print("="*80)
t_explosion = estimate_explosion_time_from_q_band()